In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *


## question 1

In [0]:
data=[
    ('Alice','Badminton,Tennis'),
    ('Bob','Tennis,Cricket'),
    ('Julie','Cricket,Carroms')
    ]
schema=['name','Hobbies']

df=spark.createDataFrame(data,schema)
df.display()

In [0]:
df1=df.withColumn('Hobbies',split(col('Hobbies'),','))
df1.display()

In [0]:
df1.withColumn('Hobbies',explode(col('Hobbies'))).display()

## question 2

In [0]:
data=[('Goa','','Ap'),('','AP',None),(None,'','Bglr')]
schema=['City1','City2','City3']

df1=spark.createDataFrame(data,schema)
df1.display()

In [0]:
# when // otherwise -> case when in sql,, if elif else in python
df1=df1.withColumn('new_column',coalesce(when(col('City1')=='',None).otherwise(col('City1')),
                                     when(col('City2')=='',None).otherwise(col('City2')),
                                     when(col('City3')=='',None).otherwise(col('City3'))))
df1.display()

In [0]:
df1.select(col('new_column').alias('Result')).display()

## question 3

In [0]:
data=[(1,'Steve'),(2,'David'),(3,'Aryan'),(4,'Shree'),(5,'Helen')
      ]
schema=['Id','Name']


data1=[(1,'pyspark',90),(1,'sql',100),(2,'sql',70),(2,'pyspark',60),(3,'sql',30),(3,'pyspark',20),(4,'sql',50),(4,'pyspark',50),(5,'sql',45),(5,'pyspark',45)]
schema1=['Id','Subject','Mark']

df1=spark.createDataFrame(data,schema)
df2=spark.createDataFrame(data1,schema1)

df1.display()
df2.display()

In [0]:
df_temp=df1.join(df2,df1.Id==df2.Id).drop(df2.Id)
df_temp.display()

In [0]:
df_temp=df_temp.groupBy('Id','Name').agg((sum('Mark')/count('*')).alias('Percentage'))
df_temp.display()

In [0]:
df_temp.select('*',
               (when(df_temp.Percentage>=70,'Distinction')
               .when((df_temp.Percentage<70) & (df_temp.Percentage >=60),'First Class')
               .when((df_temp.Percentage<60) & (df_temp.Percentage >=50),'Second Class')
               .when((df_temp.Percentage<50) & (df_temp.Percentage >=40),'Third Class')
               .otherwise('fail')).alias('Result')
               ).display()

### question 4

In [0]:
data = [
    (1, "A", 1000, "IT"),
    (2, "B", 1500, "IT"),
    (3, "C", 2500, "IT"),
    (4, "D", 3000, "HR"),
    (5, "E", 2000, "HR"),
    (6, "F", 1000, "HR"),
    (7, "G", 4000, "Sales"),
    (8, "H", 4000, "Sales"),
    (9, "I", 1000, "Sales"),
    (10, "J", 2000, "Sales")
]
schema=['EmpId','EmpName','Salary','DeptName']

df1=spark.createDataFrame(data,schema)
df1.display()

In [0]:
df1=df1.select('*',(dense_rank().over(Window.partitionBy('DeptName').orderBy(col('salary').desc()))).alias('rank'))
df1.display()

In [0]:
df1.filter(df1.rank==1).drop(col('rank')).display()

## question - 5

In [0]:
data1=[
    (100, "Raj", None, 1, "01-04-23", 50000),
    (200, "Joanne", 100, 1, "01-04-23", 4000),
    (200, "Joanne", 100, 1, "13-04-23", 4500),  
    (200, "Joanne", 100, 1, "14-04-23", 4020)   
]

schema1=['EmpId','EmpName','Mgrid','deptid','salarydt','salary']

df1=spark.createDataFrame(data1,schema1)

data2 = [
    (1, "IT"),
    (2, "HR")
]

schema2=['deptid','deptname']

df2=spark.createDataFrame(data2,schema2)
df1.display()
df2.display()

In [0]:
df=df1.withColumn('Newsaldt',to_date(col('salarydt'),'dd-MM-yy'))
df.display()

In [0]:
df3=df.join(df2,['deptid'])
df3.display()

In [0]:
df4=df3.alias('a').join(df3.alias('b'),col('a.Mgrid')==col('b.EmpId'),'left').select(col('a.deptname'),col('b.EmpName').alias('ManagerName'),
                                                                                     col('a.EmpName'),col('a.Newsaldt'),col('a.salary'))
df4.display()

In [0]:
df5=df4.groupby('deptname','ManagerName','EmpName',year('Newsaldt').alias('year'),date_format('Newsaldt','MMMM').alias('Month')).sum('salary')
df5.display()

### question-6

In [0]:
dbutils.fs.ls('/Volumes/databricks_tutorial/default/my_volume')

In [0]:
df=spark.read.format('csv').option('header',True).option('inferSchema',True).load('/Volumes/databricks_tutorial/default/my_volume/BigMart Sales (1).csv')
df.display()

In [0]:
# df.rdd.getNumPartitions()
df.selectExpr("spark_partition_id() as partition_id") \
  .distinct() \
  .count()

In [0]:
df.repartition(100)

In [0]:
df.selectExpr("spark_partition_id() as partition_id") \
  .distinct() \
  .show()

In [0]:
df=df.select(spark_partition_id().alias('partId')).groupBy('partId').count()
df.display()

In [0]:
# spark.read.format('csv').option('header',True).option('inferSchema',True).load('')
# df.display()
# df.rdd.getNumPartitions()
# df.repartition(10)
# df.rdd.getNumPartitions()
# df.select(spark_partition_id().alias('t')).groupBy('t').count()


### question 7

In [0]:
# permissive
# failfast
# dropmalformed


# df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").option("mode", "PERMISSIVE").load("/FileStore/tables/retail_data.csv")\
# .display()

data=[(1,'ajay',23,'botany'),('2uuu','bala',24,'banaras'),(3,'chandu',25,'banglore'),('4ddd','dinesh',26,'bhopal')]
schema='''
    Id String,
    Name string,
    Age int,
    Subject string 
'''
df=spark.createDataFrame(data,schema)
df.display()

![image_1780397426451.png](./image_1780397426451.png "image_1780397426451.png")